In [2]:
import pandas as pd
import psycopg2

sheet_curr = '1wxR3iYna86qrdViwHjUPzHuw6bCNeMLb72M25hpUHYk'
sheet_archive = '1PxNYGMXaVrRqI0uyMQF46K7nDEG16WnDoKrFyI_qrvE'
gid_matches = '2141931777'
gid_deck = '590005429'

In [3]:
with open("credentials.txt", "r") as file:
    credentials = [line.strip() for line in file]

In [ ]:
def conn(query):
    try:
        conn = psycopg2.connect(
            host=credentials[0],
            port=credentials[1],
            user=credentials[2],
            password=credentials[3],
            database=credentials[4]
        )
        cursor = conn.cursor()

        cursor.execute(query)

        conn.commit()
    except psycopg2.Error as e:
        print('Error:', e)
    finally:
        if conn:
            cursor.close()
            conn.close()

In [1]:
def delete_table(TABLE):
    query = f'DROP TABLE IF EXISTS {TABLE} CASCADE'
    conn(query)

In [38]:
def create_new_tables():
    create_valid_decks_query = """
    CREATE TABLE IF NOT EXISTS VALID_DECKS (
        FORMAT VARCHAR(30),
        ARCHETYPE VARCHAR(30),
        SUBARCHETYPE VARCHAR(30),
        DECK_ID INT PRIMARY KEY
    );
    """
    create_valid_event_types_query = """
    CREATE TABLE IF NOT EXISTS VALID_EVENT_TYPES (
        FORMAT VARCHAR(30),
        EVENT_TYPE VARCHAR(30),
        EVENT_TYPE_ID INT PRIMARY KEY
    );
    """
    create_events_query = """
    CREATE TABLE IF NOT EXISTS EVENTS (
        EVENT_ID INT PRIMARY KEY,
        EVENT_DATE DATE,
        EVENT_TYPE_ID INT,
        FOREIGN KEY (EVENT_TYPE_ID) REFERENCES VALID_EVENT_TYPES(EVENT_TYPE_ID)
    );
    """
    create_matches_query = """
    CREATE TABLE IF NOT EXISTS MATCHES (
        MATCH_ID BIGINT,
        P1 VARCHAR(30),
        P2 VARCHAR(30),
        P1_WINS INT,
        P2_WINS INT,
        MATCH_WINNER VARCHAR(2),
        P1_DECK_ID INT,
        P2_DECK_ID INT,
        P1_NOTE VARCHAR(30),
        P2_NOTE VARCHAR(30),
        EVENT_ID INT,
        PRIMARY KEY (MATCH_ID, P1),
        FOREIGN KEY (P1_DECK_ID) REFERENCES VALID_DECKS(DECK_ID),
        FOREIGN KEY (P2_DECK_ID) REFERENCES VALID_DECKS(DECK_ID),
        FOREIGN KEY (EVENT_ID) REFERENCES EVENTS(EVENT_ID)
    );
    """
    create_loadreports_query = """
    CREATE TABLE IF NOT EXISTS LOAD_REPORTS (
        LOAD_RPT_ID BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 100000000) PRIMARY KEY,
        CHANGE_CODE VARCHAR(3),
        RECORDS_ADD INT,
        RECORDS_MOD INT,
        RECORDS_DEL INT,
        PROC_DT DATE
    );
    """
    create_rejections_query = """
    CREATE TABLE IF NOT EXISTS REJECTIONS (
        REJ_ID BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 2000000000) PRIMARY KEY,
        LOAD_RPT_ID INT,
        TABLE_NAME VARCHAR(30),
        COL_NAMES VARCHAR(250),
        COL_VALUES VARCHAR(250),
        REJ_TXT VARCHAR(250),
        FOREIGN KEY (LOAD_RPT_ID) REFERENCES LOAD_REPORTS(LOAD_RPT_ID)
    );
    """
    create_fkey_indexes = """
    CREATE INDEX idx_events_event_type_id ON EVENTS(EVENT_TYPE_ID);
    CREATE INDEX idx_matches_p1_deck_id ON MATCHES(P1_DECK_ID);
    CREATE INDEX idx_matches_p2_deck_id ON MATCHES(P2_DECK_ID);
    CREATE INDEX idx_matches_event_id ON MATCHES(EVENT_ID);
    CREATE INDEX idx_rejections_load_rpt_id ON REJECTIONS(LOAD_RPT_ID);
    """
    try:
        conn(create_valid_decks_query)
        conn(create_valid_event_types_query)
        conn(create_events_query)
        conn(create_matches_query)
        conn(create_loadreports_query)
        conn(create_rejections_query)
        conn(create_fkey_indexes)
    except:
        print('Exception. No tables created.')

In [40]:
delete_table('VALID_DECKS')
delete_table('VALID_EVENT_TYPES')
delete_table('EVENTS')
delete_table('MATCHES')
delete_table('LOAD_REPORTS')
delete_table('REJECTIONS')

In [41]:
create_new_tables()